# The Selective Pirate: Matryoshka Attribution over a Fixed LoRA Delta

[Matryoshka Attribution](https://arxiv.org/html/2609.25518v1) (September 2026, algorithm 1 and appendix J) asks which neurons of a fine-tuning update carry the behavior it taught. Train a fine-tuning delta, freeze it, and learn a score per output row that survives a randomized sparsity budget. Rows that keep paying off at small budgets rank first, and the masks nest like the dolls in the method's name.

Here a model learns an invented pirate catchphrase, "By my pocket kraken!", through a LoRA delta. The delta is then fixed while Fine-Tuning Scheduler schedules the score tensors. Each ranking is evaluated at nested row budgets on held-out questions, and the question is how much of the catchphrase survives per row, and what it costs neutral prompts.

Three scopes, kept distinct throughout. The **reproduced method** is fixed-delta attribution under randomized budgets, implemented independently because upstream declares no reuse license. The **bounded transfer** is a local twenty-question corpus on [Qwen3-4B](https://huggingface.co/Qwen/Qwen3-4B), over the query and MLP-up projections of the last four blocks, not the paper's Alpaca rewrite corpus or model families. The **extension** is phased score training, which the paper does not do: score tensors are thawed in phases by name regex, then by venngram role, layer and component addresses.

## Setup

The parameters keep this executed artifact download-free: a small randomly initialized decoder, labeled a smoke model wherever its numbers appear. **Smoke numbers demonstrate the machinery, not the method.** The [pretrained results page](../../selective_pirate_results.rst) records the Qwen3-4B runs and their limits. Set `MODEL = "qwen3"` and `DEVICE = "cuda"` to rerun that path here under your host's GPU lease.

In [ ]:
SEED = 1
DONOR_STEPS = 20
STEPS_PER_EPOCH = 4
N_EVAL = 2
MAX_NEW_TOKENS = 16
FRACTIONS = (0.0, 0.1, 0.5, 1.0)
MODEL = "smoke"
DEVICE = "cpu"

In [ ]:
import warnings

warnings.filterwarnings("ignore", message=".*addresses.*experimental")

from fts_examples.tunable_views.pirate import PIRATE_PHRASE, PirateConfig, corpus
from fts_examples.tunable_views.pirate_run import plot_budget_curves, run

values = dict(seed=SEED, donor_steps=DONOR_STEPS, steps_per_epoch=STEPS_PER_EPOCH, n_eval=N_EVAL,
              max_new_tokens=MAX_NEW_TOKENS, fractions=FRACTIONS, device=DEVICE)
config = PirateConfig.qwen3(**values) if MODEL == "qwen3" else PirateConfig(**values)
print("run config fingerprint (identical for both interfaces):", config.fingerprint())

## The data

Each question has a neutral answer and a pirate rendering of it. The pirate prompt asks for the pirate answer, which appends the catchphrase. Labels cover only the completion and its end token, and the split is by unique question, so no held-out question reaches donor or mask training. Pirate greetings are a poor signal on a pretrained chat model, which already answers "Ahoy, matey!" when addressed that way, so the invented phrase is the behavior being measured.

In [ ]:
train, held_out = corpus(config.n_eval, config.seed)
print(f"{len(train)} training questions, {len(held_out)} held out, disjoint by question")
example = held_out[0]
print("pirate prompt:   ", example.prompt(True))
print("pirate response: ", example.response(True))
print("neutral prompt:  ", example.prompt(False))
print("neutral response:", example.response(False))
print("measured phrase: ", PIRATE_PHRASE)

## A donor, then a frozen delta

The donor trains rank-scaled LoRA adapters by name regex with every base weight frozen. Its factors are then frozen too, and each output row $i$ of a projection gets a gate on its delta:

$$
y_i = (W x)_i + m_i (\Delta x)_i
$$

where $\Delta$ is the frozen update and $W$ the frozen base. At $m_i = 1$ for all rows the model is the donor, and at $m_i = 0$ it is the base. `MaskedDeltaLinear` keeps the two factor multiplications rather than merging them, because a merged dense delta does not reproduce the donor's BF16 logits.

During score training a single threshold $\tau$ across all eligible rows meets a sampled budget $k$:

$$
m_i = \sigma\left( \frac{s_i - \tau}{T} \right), \qquad \sum_i m_i = k
$$

Backward differentiates that constraint, including the coupling it creates between scores: a score held frozen can still have a changing mask. Evaluation replaces the soft mask with the top $k$ scores, so hard masks nest across budgets. Budgets are drawn uniformly from the interior integers, by a local generator, so both interfaces see the same sequence.

## The phased schedule, on both interfaces

Fine-Tuning Scheduler schedules tensors, so a phase selects whole score tensors, and the explicit wrapper's row list names the coordinates. Phase 0 uses a regex, for one known model. Phases 1 and 2 use addresses (`role:adapter:mask & component:attn.q & layers:-2:` and the remaining attention scores), which resolve through the model's view on any mapped architecture. The one-phase control trains every score for the same steps and the same budget sequence.

`run` drives the PyTorch API and Lightning from one definition, sequentially, then checks they agree. The next cell captures their logs: `run_log.show()` prints them.

In [ ]:
%%capture run_log
result = run(config)

In [ ]:
paired = result["paired"]
for stage in ("donor", "phased", "one_phase"):
    check = paired[stage]
    assert check["initial_state_equal"] and check["final_state_equal"] and check["phases_equal"]
    print(f"{stage:10s} {check['steps']:3d} steps, max loss difference {check['max_loss_difference']}")
assert paired["evaluation"] == {"max_metric_difference": 0.0, "responses_differ": []}
print("Lightning and PyTorch: identical states, phases, losses and evaluations")

report = result["results"]["pytorch"]
for epoch in report["curves"]["phased"]["epochs"]:
    print(f"epoch {epoch['epoch']} (phase {epoch['depth']}): {len(epoch['trainable'])} score tensors")
    for name in epoch["trainable"]:
        print("   ", name)

Lightning's strategy teardown returns the module to CPU after `fit`. The runner moves it back, and `evaluate` refuses any other device, because BF16 arithmetic differs by device even over identical parameter bytes. Evaluations are paired only because of that step.

## What the scores address

Every eligible row belongs to a projection's output axis. For an MLP up projection, a row is an intermediate neuron. Row fractions are fractions of this eligible subset, not of the model's parameters.

In [ ]:
for item in report["coordinates"]:
    print(f"{item['score_name']:45s} layer {item['layer']} {item['component']:7s} "
          f"{item['axis']} rows {item['eligible_range']}")
print("eligible rows:", report["eligible_rows"], "; fixed base and delta verified:", report["fixed_base_and_delta"])
print("donor endpoint max logit error:", report["donor_endpoint_max_abs_logit_error"])

## Rankings at matched budgets

Five rankings share the donor and row universe. `phased` and `one_phase` are learned. `random` is a seeded permutation. `gradient_base` and `gradient_full` rank by negative gate gradients at the base and donor endpoints, with as many backward passes as score training took.

The columns separate three things that are easy to conflate. Held-out **loss** is the loss on the pirate completions. **Phrase** is the fraction of greedy pirate responses containing the catchphrase, with **off** the same on neutral prompts. **Neutral loss** and **KL** (next-token divergence from the base on neutral prompts) are retention proxies, not a capability score. The phrase rate says nothing about whether the answer is right.

In [ ]:
print(f"{'ranking':14s} {'fraction':>8s} {'rows':>6s} {'loss':>8s} {'phrase':>6s} {'off':>5s} {'neutral':>8s} {'KL':>8s}")
for name, points in report["rankings"].items():
    for point in points:
        e = point["evaluation"]
        print(f"{name:14s} {point['fraction']:8.2f} {point['budget']:6d} {e['pirate_prompt_loss']:8.3f} "
              f"{e['phrase_rate']:6.2f} {e['off_target_phrase_rate']:5.2f} {e['neutral_response_loss']:8.3f} "
              f"{e['neutral_next_token_kl']:8.4f}")

In [ ]:
plot_budget_curves([report], title=f"Selective pirate rankings ({MODEL})");

## Which rows, and how much the rankings agree

Each point records its selected row count per projection and the first few row indices, so a selection is a physical statement about the model rather than a percentage. Jaccard overlap between rankings shows whether two procedures found the same rows.

In [ ]:
point = next(p for p in report["rankings"]["phased"] if 0 < p["fraction"] < 1)
print(f"phased selection at {point['budget']} rows:")
for path, selected in point["selected"].items():
    print(f"  {path:40s} {selected['count']:6d} rows, first {selected['first_rows'][:6]}")
for pair, values in report["row_overlap"].items():
    print(f"{pair:32s}", {budget: round(overlap, 3) for budget, overlap in values.items()})

## Read the responses

Rates compress too much. Always look at what the model said, at the base, the donor and the smallest budget.

In [ ]:
print("base pirate:  ", report["base"]["responses"]["pirate"])
print("donor pirate: ", report["donor_evaluation"]["responses"]["pirate"])
print("donor neutral:", report["donor_evaluation"]["responses"]["neutral"])
for name, points in report["rankings"].items():
    point = next(p for p in points if p["budget"] > 0)
    print(f"{name} at {point['budget']} rows:", point["evaluation"]["responses"]["pirate"])

## Reproduce the pretrained runs

Run the same harness from a shell, under your host's GPU lease. Results are written after each completed stage, so a failure keeps earlier evidence.

```bash
python -m fts_examples.tunable_views.pirate_run --qwen3 --output pirate_paired.json
python -m fts_examples.tunable_views.pirate_run --qwen3 --interfaces pytorch --seed 2 --n-eval 4 \
    --fractions 0,0.01,0.02,0.05,0.1,0.25,0.5,1 --output pirate_sweep_s2.json
```

The local corpus, lexical metrics, the eligible projections and phased score training differ from the paper. Down projections and other output bases are not attributed, since the row wrapper cannot claim their column support. Nothing here localizes the behavior causally, and low loss at a small budget does not show that those rows carry useful generalization.